# US-03 — Validation des dépendances Spark

**EPIC-01 — Infrastructure** · Responsable : Zakaria

| Ticket | Critère | Section |
|---|---|---|
| SBD-55 | Les versions de Spark, Scala, Java, PySpark, Kafka, Delta Lake et GraphFrames sont documentées | 1 et 6 |
| SBD-56 | Spark peut lire des événements depuis Kafka | 2 |
| SBD-57 | Spark peut lire et écrire des fichiers Parquet dans MinIO | 3 |
| SBD-58 | Delta Lake est configuré et une opération d'écriture/lecture fonctionne | 4 |
| SBD-59 | GraphFrames est installé et un graphe simple peut être construit avec PySpark | 5 |
| SBD-60 | GraphX (Scala) : voir `graph/graphx/us03_graphx_test.scala` | — |

Les dépendances sont déclarées dans `infrastructure/spark/spark-defaults.conf`, monté dans le conteneur Jupyter : elles sont téléchargées par le Driver au démarrage de la session puis transmises aux Executors.

> Après une modification de `spark-defaults.conf`, redémarrer le kernel (**Kernel > Restart Kernel**) : les dépendances ne sont lues qu'au démarrage de la JVM du Driver.

## 0. Création de la SparkSession

Les identifiants MinIO ne sont pas écrits dans le code : ils proviennent du fichier `.env`, transmis au conteneur Jupyter par Docker Compose.

In [1]:
import os
from pyspark.sql import SparkSession

builder = (
    SparkSession.builder
    .appName("SahlPay-US03-Dependances")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.cores", "2")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
)

# Identifiants MinIO (SBD-57), lus dans les variables d'environnement du conteneur
if os.environ.get("MINIO_ROOT_USER"):
    builder = (
        builder
        .config("spark.hadoop.fs.s3a.access.key", os.environ["MINIO_ROOT_USER"])
        .config("spark.hadoop.fs.s3a.secret.key", os.environ["MINIO_ROOT_PASSWORD"])
    )

spark = builder.getOrCreate()
sc = spark.sparkContext

print("Application :", sc.appName)
print("Master      :", sc.master)
print("Packages    :")
for p in sc.getConf().get("spark.jars.packages", "aucun").split(","):
    print("  -", p)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/spark/.ivy2/cache
The jars for the packages stored in: /home/spark/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
io.delta#delta-spark_2.12 added as a dependency
io.graphframes#graphframes-spark3_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5a4761cf-7bce-4447-88c3-400d1f9106e2;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.7 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.7 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.

Application : SahlPay-US03-Dependances
Master      : spark://spark-master:7077
Packages    :
  - org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.7
  - org.apache.hadoop:hadoop-aws:3.3.4
  - io.delta:delta-spark_2.12:3.3.2
  - io.graphframes:graphframes-spark3_2.12:0.9.2


## 1. Versions de la plateforme (SBD-55)

In [2]:
import sys

jvm = sc._jvm
print("Spark   :", spark.version)
print("Scala   :", jvm.scala.util.Properties.versionNumberString())
print("Java    :", jvm.java.lang.System.getProperty("java.version"))
print("Python  :", sys.version.split()[0])
import pyspark
print("PySpark :", pyspark.__version__)

Spark   : 3.5.7
Scala   : 2.12.18
Java    : 17.0.16
Python  : 3.10.12
PySpark : 3.5.7


## 2. Lecture d'événements depuis Kafka (SBD-56)

Lecture **batch** du topic de test `us03-kafka-test` (3 messages envoyés avec `kafka-console-producer.sh`).
Kafka renvoie des colonnes techniques (`key`, `value`, `topic`, `partition`, `offset`, `timestamp`) ; le message lui-même est dans `value`, au format binaire.

In [3]:
df_kafka = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "us03-kafka-test")
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
)

df_kafka.printSchema()

(
    df_kafka
    .selectExpr("topic", "partition", "offset", "timestamp", "CAST(value AS STRING) AS value")
    .show(truncate=False)
)

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



26/11/07 19:49:36 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                

+---------------+---------+------+-----------------------+-----------------------------------------------------------------------------------+
|topic          |partition|offset|timestamp              |value                                                                              |
+---------------+---------+------+-----------------------+-----------------------------------------------------------------------------------+
|us03-kafka-test|0        |0     |2026-11-07 18:07:34.357|{"tx_id":"TX-TEST-001","status":"COMPLETED","amount_mad":150.0,"city":"Rabat"}     |
|us03-kafka-test|0        |1     |2026-11-07 18:07:34.373|{"tx_id":"TX-TEST-002","status":"REJECTED","amount_mad":2500.0,"city":"Casablanca"}|
|us03-kafka-test|0        |2     |2026-11-07 18:07:34.374|{"tx_id":"TX-TEST-003","status":"INITIATED","amount_mad":45.5,"city":"Fes"}        |
+---------------+---------+------+-----------------------+-----------------------------------------------------------------------------------+

Conversion du JSON en colonnes avec un schéma explicite :

In [4]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import DoubleType, StringType, StructField, StructType

schema = StructType([
    StructField("tx_id", StringType()),
    StructField("status", StringType()),
    StructField("amount_mad", DoubleType()),
    StructField("city", StringType()),
])

events = (
    df_kafka
    .select(from_json(col("value").cast("string"), schema).alias("e"))
    .select("e.*")
)

events.show()
print("Événements lus depuis Kafka :", events.count())

26/11/07 19:49:48 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/11/07 19:49:53 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+-----------+---------+----------+----------+
|      tx_id|   status|amount_mad|      city|
+-----------+---------+----------+----------+
|TX-TEST-001|COMPLETED|     150.0|     Rabat|
|TX-TEST-002| REJECTED|    2500.0|Casablanca|
|TX-TEST-003|INITIATED|      45.5|       Fes|
+-----------+---------+----------+----------+



[Stage 2:>                                                          (0 + 1) / 1]

Événements lus depuis Kafka : 3


## 3. Parquet dans MinIO (SBD-57)

Spark accède à MinIO par le connecteur **S3A** (`hadoop-aws`) : les chemins commencent par `s3a://<bucket>/`.
Les fichiers de test sont écrits sous `lake/_tests/us03/` pour ne pas mélanger avec les couches Bronze, Silver et Gold.

In [5]:
parquet_path = "s3a://lake/_tests/us03/parquet"

events.write.mode("overwrite").parquet(parquet_path)

events_parquet = spark.read.parquet(parquet_path)
events_parquet.show()
print("Lignes relues depuis MinIO :", events_parquet.count())

26/11/07 19:49:55 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/11/07 19:49:56 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                

+-----------+---------+----------+----------+
|      tx_id|   status|amount_mad|      city|
+-----------+---------+----------+----------+
|TX-TEST-001|COMPLETED|     150.0|     Rabat|
|TX-TEST-002| REJECTED|    2500.0|Casablanca|
|TX-TEST-003|INITIATED|      45.5|       Fes|
+-----------+---------+----------+----------+

Lignes relues depuis MinIO : 3


## 4. Delta Lake (SBD-58)

Une table Delta = des fichiers Parquet + un journal de transactions (`_delta_log/`).
Test : écriture (version 0), ajout d'une ligne (version 1), lecture, historique et *time travel*.

In [6]:
from delta.tables import DeltaTable

delta_path = "s3a://lake/_tests/us03/delta"

# Version 0 : écriture initiale
events.write.format("delta").mode("overwrite").save(delta_path)

# Version 1 : ajout d'un événement
extra = spark.createDataFrame([("TX-TEST-004", "COMPLETED", 99.0, "Tanger")], schema)
extra.write.format("delta").mode("append").save(delta_path)

spark.read.format("delta").load(delta_path).show()

26/11/07 19:50:06 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/11/07 19:50:17 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                

+-----------+---------+----------+----------+
|      tx_id|   status|amount_mad|      city|
+-----------+---------+----------+----------+
|TX-TEST-001|COMPLETED|     150.0|     Rabat|
|TX-TEST-002| REJECTED|    2500.0|Casablanca|
|TX-TEST-003|INITIATED|      45.5|       Fes|
|TX-TEST-004|COMPLETED|      99.0|    Tanger|
+-----------+---------+----------+----------+



In [7]:
table = DeltaTable.forPath(spark, delta_path)
table.history().select("version", "timestamp", "operation").show(truncate=False)

v0 = spark.read.format("delta").option("versionAsOf", 0).load(delta_path).count()
v1 = spark.read.format("delta").load(delta_path).count()
print("Lignes en version 0 :", v0)
print("Lignes en version 1 :", v1)

+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|3      |2026-11-07 19:50:23|WRITE    |
|2      |2026-11-07 19:50:20|WRITE    |
|1      |2026-11-07 19:26:56|WRITE    |
|0      |2026-11-07 19:26:52|WRITE    |
+-------+-------------------+---------+



[Stage 39:=================================================>      (44 + 4) / 50]

Lignes en version 0 : 3
Lignes en version 1 : 4


## 5. GraphFrames (SBD-59)

Petit graphe inspiré de la Partie 6 : trois comptes utilisent le même appareil, et un compte transfère à un autre.

In [8]:
from graphframes import GraphFrame

vertices = spark.createDataFrame(
    [("W-1", "compte"), ("W-2", "compte"), ("W-3", "compte"), ("D-1", "appareil")],
    ["id", "type"],
)

edges = spark.createDataFrame(
    [
        ("W-1", "D-1", "USES_DEVICE"),
        ("W-2", "D-1", "USES_DEVICE"),
        ("W-3", "D-1", "USES_DEVICE"),
        ("W-1", "W-2", "TRANSFERS_TO"),
    ],
    ["src", "dst", "relationship"],
)

g = GraphFrame(vertices, edges)

print("Sommets :", g.vertices.count(), "| Arêtes :", g.edges.count())
g.inDegrees.orderBy("inDegree", ascending=False).show()

Sommets : 4 | Arêtes : 4
+---+--------+
| id|inDegree|
+---+--------+
|D-1|       3|
|W-2|       1|
+---+--------+



Recherche de motif : paires de comptes qui utilisent le même appareil.

In [9]:
from pyspark.sql.functions import col

shared = (
    g.find("(a)-[e1]->(d); (b)-[e2]->(d)")
    .filter("d.type = 'appareil' AND e1.relationship = 'USES_DEVICE' AND e2.relationship = 'USES_DEVICE' AND a.id < b.id")
    .select(col("a.id").alias("compte_1"), col("b.id").alias("compte_2"), col("d.id").alias("appareil"))
)

shared.show()

+--------+--------+--------+
|compte_1|compte_2|appareil|
+--------+--------+--------+
|     W-1|     W-2|     D-1|
|     W-2|     W-3|     D-1|
|     W-1|     W-3|     D-1|
+--------+--------+--------+



## 6. Versions des bibliothèques installées (SBD-55)

In [10]:
from importlib.metadata import version

print("delta-spark (Python)    :", version("delta-spark"))
print("graphframes-py (Python) :", version("graphframes-py"))
print("Packages JVM            :")
for p in sc.getConf().get("spark.jars.packages").split(","):
    print("  -", p)

delta-spark (Python)    : 3.3.2
graphframes-py (Python) : 0.9.2
Packages JVM            :
  - org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.7
  - org.apache.hadoop:hadoop-aws:3.3.4
  - io.delta:delta-spark_2.12:3.3.2
  - io.graphframes:graphframes-spark3_2.12:0.9.2


## 7. Libération des ressources

In [12]:
spark.stop()